In [1]:
import numpy as np
from tqdm import tqdm
from PIL import Image
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, accuracy_score


In [2]:
DATA_DIR = Path("/content/drive/MyDrive/PJ/archive")
SEED = 42


In [3]:
X = []
y = []

classes = sorted([p.name for p in DATA_DIR.iterdir() if p.is_dir()])
class_to_idx = {cls: idx for idx, cls in enumerate(classes)}

for cls in classes:
    class_dir = DATA_DIR / cls
    for img_path in tqdm(list(class_dir.glob("*")), desc=f"Cargando {cls}"):
        try:
            img = Image.open(img_path).convert("RGB")
            img = img.resize(IMG_SIZE)
            img = np.array(img, dtype=np.float32) / 255.0
            X.append(img.flatten())
            y.append(class_to_idx[cls])
        except Exception:
            continue


Cargando muskmelon: 100%|██████████| 2078/2078 [01:09<00:00, 29.94it/s]


In [ ]:
X = np.array(X, dtype=np.float32)
y = np.array(y)

print("X shape:", X.shape)
print("y shape:", y.shape)


X shape: (31201, 6912)
y shape: (31201,)


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=SEED
)

# Subsample para entrenamiento
N_TRAIN = 3000
X_train = X_train[:N_TRAIN]
y_train = y_train[:N_TRAIN]


In [ ]:
from sklearn.decomposition import PCA
from sklearn.neighbors import KNeighborsClassifier # Import KNeighborsClassifier here

pca = PCA(n_components=100, random_state=SEED)
X_train_pca = pca.fit_transform(X_train)
X_test_pca = pca.transform(X_test)

# Initialize knn before using it
knn = KNeighborsClassifier(
    n_neighbors=5,
    weights="distance",
    metric="euclidean",
    n_jobs=-1
)

knn.fit(X_train_pca, y_train)


KNeighborsClassifier(metric='euclidean', n_jobs=-1, weights='distance')

In [ ]:
knn = KNeighborsClassifier(
    n_neighbors=5,
    weights="distance",
    metric="euclidean",
    n_jobs=-1
)

knn.fit(X_train, y_train)


KNeighborsClassifier(metric='euclidean', n_jobs=-1, weights='distance')

In [ ]:
y_pred = knn.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=classes, labels=np.arange(len(classes))))


Accuracy: 0.7183143726966832
              precision    recall  f1-score   support

       Apple       0.00      0.00      0.00         0
      Banana       0.90      0.63      0.74       606
   Carambola       0.58      0.78      0.67       416
       Guava       0.00      0.00      0.00         0
        Kiwi       0.00      0.00      0.00         0
       Mango       0.74      0.71      0.72       831
      Orange       0.90      0.56      0.69       603
       Peach       0.60      0.76      0.67       526
        Pear       0.58      0.79      0.67       602
   Persimmon       0.77      0.76      0.76       414
      Pitaya       0.74      0.68      0.71       500
        Plum       0.87      0.79      0.83       460
 Pomegranate       0.99      0.75      0.85       433
    Tomatoes       0.89      0.67      0.76       434
   muskmelon       0.52      0.83      0.64       416

    accuracy                           0.72      6241
   macro avg       0.61      0.58      0.58      62

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/me

DL

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models


In [ ]:
DATA_DIR = "/content/drive/MyDrive/PJ/archive"
IMG_SIZE = (64, 64)
BATCH_SIZE = 32
SEED = 42

train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset="training",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset="validation",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)


Found 70549 files belonging to 15 classes.
Using 56440 files for training.
Found 70549 files belonging to 15 classes.
Using 14109 files for validation.


In [ ]:
train_ds = train_ds.map(lambda x, y: (x / 255.0, y))
val_ds = val_ds.map(lambda x, y: (x / 255.0, y))


In [ ]:
model = models.Sequential([
    layers.Conv2D(16, 3, activation="relu", input_shape=(*IMG_SIZE, 3)),
    layers.MaxPooling2D(),

    layers.Conv2D(32, 3, activation="relu"),
    layers.MaxPooling2D(),

    layers.Flatten(),
    layers.Dense(64, activation="relu"),
    layers.Dense(15, activation="softmax")
])


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)


In [ ]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10
)


Epoch 1/10
1764/1764 ━━━━━━━━━━━━━━━━━━━━ 5914s 3s/step - accuracy: 0.7178 - loss: 0.9219 - val_accuracy: 0.9296 - val_loss: 0.2025
Epoch 2/10
1764/1764 ━━━━━━━━━━━━━━━━━━━━ 723s 410ms/step - accuracy: 0.9469 - loss: 0.1618 - val_accuracy: 0.9496 - val_loss: 0.1340
Epoch 3/10
1764/1764 ━━━━━━━━━━━━━━━━━━━━ 717s 407ms/step - accuracy: 0.9692 - loss: 0.0892 - val_accuracy: 0.9658 - val_loss: 0.0915
Epoch 4/10
1764/1764 ━━━━━━━━━━━━━━━━━━━━ 722s 409ms/step - accuracy: 0.9795 - loss: 0.0599 - val_accuracy: 0.9648 - val_loss: 0.1035
Epoch 5/10
1764/1764 ━━━━━━━━━━━━━━━━━━━━ 1107s 627ms/step - accuracy: 0.9855 - loss: 0.0419 - val_accuracy: 0.9639 - val_loss: 0.1109
Epoch 6/10
1764/1764 ━━━━━━━━━━━━━━━━━━━━ 724s 410ms/step - accuracy: 0.9903 - loss: 0.0278 - val_accuracy: 0.9603 - val_loss: 0.1315
Epoch 7/10
1764/1764 ━━━━━━━━━━━━━━━━━━━━ 730s 414ms/step - accuracy: 0.9908 - loss: 0.0282 - val_accuracy: 0.9630 - val_loss: 0.1312
Epoch 8/10
1764/1764 ━━━━━━━━━━━━━━━━━━━━ 730s 414ms/step - acc

In [ ]:
loss, acc = model.evaluate(val_ds)
print("Accuracy:", acc)


441/441 ━━━━━━━━━━━━━━━━━━━━ 115s 260ms/step - accuracy: 0.9637 - loss: 0.1455
Accuracy: 0.9625061750411987
